# Notebook 01 — CNN Baseline + MobileNetV2 Student + ResNet50 Teacher

**NeuroDriver CNN ADAS Colombia** — Hito de la Fase 1 (actualizado 2026-09-22).

Diseñado para ejecutarse en Google Colab (montando Drive / clonando el repositorio), pero portable a
Windows/Linux local. El entrenamiento del ResNet50 Teacher y la Knowledge Distillation **no** se
ejecutan en este notebook — ver las Secciones 14-16 para lo que está preparado y lo que queda
pendiente.

Los modelos se entrenan sobre dos targets binarios independientes (`has_vehicle`, `has_pedestrian`)
mediante `BinaryCrossentropy(from_logits=True)`, no softmax de 4 clases — los datos reales mostraron
un desbalance severo para los frames puros de PEDESTRIAN. Ver `docs/decisions_log.md`.


## 1. Verificación de entorno / GPU

In [ ]:
import subprocess, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "configs").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts" / "00_check_environment.py")])


## 2. Semillas deterministas

In [ ]:
from neurodriver_cnn.utils.seed import set_global_seed
from neurodriver_cnn.config import load_dataset_config, load_training_config

SEED = load_dataset_config(PROJECT_ROOT)["seed"]
set_global_seed(SEED)
print(f"Semilla configurada: {SEED}")


## 3. Raíz de proyecto / datos configurable

Sin paths personales ni hardcodeados de Google Drive — sobrescribe `DATA_ROOT` abajo solo si tus
datos están fuera de `PROJECT_ROOT/data`.


In [ ]:
DATA_ROOT = PROJECT_ROOT / "data"
MANIFEST_PATH = DATA_ROOT / "processed" / "manifests" / "experiment_manifest.csv"
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"MANIFEST_PATH = {MANIFEST_PATH}")


## 4. Carga del Common Manifest + validación de etiquetas

In [ ]:
import pandas as pd
from neurodriver_cnn.data.manifest import validate_manifest_invariants

if MANIFEST_PATH.exists():
    manifest_df = pd.read_csv(MANIFEST_PATH)
    violations = validate_manifest_invariants(manifest_df)
    print(f"Se cargaron {len(manifest_df)} filas.")
    if violations:
        print("VIOLACIONES:")
        for v in violations:
            print(f" - {v}")
    else:
        print("[OK] El manifest pasó la validación de invariantes.")
else:
    manifest_df = None
    print("PENDING: no se encontró experiment_manifest.csv.")
    print("Ejecuta scripts/00-04 después de colocar el .tar de BDD100K (ver docs/bdd100k_setup.md).")


## 5. Pipeline de tf.data (TRAIN/VAL/TEST)

**Targets multi-label:** cada ejemplo produce `(image, [has_vehicle, has_pedestrian])` como un vector
float32 de 2 elementos, coincidiendo con la salida de dos logits de los modelos. **`prefetch`
intencionalmente no se llama aquí** — se aplica una sola vez, como el último paso de cada pipeline,
en la Sección 6, después de adjuntar el Data Augmentation exclusivo de train.


In [ ]:
import tensorflow as tf

IMAGE_SIZE = tuple(load_dataset_config(PROJECT_ROOT)["image_size"])
BATCH_SIZE = load_training_config(PROJECT_ROOT)["batch_size"]


def load_image(path, targets):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMAGE_SIZE)
    return image, targets


def make_dataset(df: pd.DataFrame, split: str, shuffle: bool) -> tf.data.Dataset:
    split_df = df[df["split"] == split]
    paths = split_df["image_path"].tolist()
    targets = split_df[["has_vehicle", "has_pedestrian"]].astype("float32").values
    ds = tf.data.Dataset.from_tensor_slices((paths, targets))
    if shuffle:
        ds = ds.shuffle(buffer_size=max(1, len(paths)), seed=SEED)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    return ds  # sin .prefetch() aquí — se aplica al final, en la Sección 6, después del augmentation


if manifest_df is not None:
    train_ds = make_dataset(manifest_df, "TRAIN", shuffle=True)
    val_ds = make_dataset(manifest_df, "VALIDATION", shuffle=False)
    test_ds = make_dataset(manifest_df, "TEST", shuffle=False)
    print("[OK] Pipelines de tf.data construidos (multi-label de 2 targets).")
else:
    train_ds = val_ds = test_ds = None
    print("PENDING: no se pueden construir los pipelines de tf.data sin un manifest.")


## 6. Data Augmentation exclusivo de TRAIN + prefetch final

El Data Augmentation (flip horizontal, brillo/contraste/zoom moderados) se aplica **únicamente** a
`train_ds`, después del split (sin fuga de datos) y antes del `prefetch` final. **Nunca** flip
vertical ni rotación de 90/180 grados — las etiquetas actuales no codifican semántica de
izquierda/derecha que un flip horizontal violaría, pero un flip vertical o una rotación de la vía
producirían escenas de manejo físicamente absurdas.

`prefetch(AUTOTUNE)` se aplica **al final, a los tres pipelines** (train/val/test) — validation y
test nunca reciben augmentation, solo prefetch.


In [ ]:
augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomBrightness(0.1),
        tf.keras.layers.RandomContrast(0.1),
        tf.keras.layers.RandomZoom(0.1),
    ],
    name="train_augmentation",
)


def augment(image, targets):
    return augmentation(image, training=True), targets


if train_ds is not None:
    # Augmentation: solo TRAIN, aplicado antes del prefetch final de abajo.
    train_ds = train_ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)

    # prefetch es el ÚLTIMO paso de cada pipeline. val_ds/test_ds se saltan
    # el augmentation por completo y van directo a prefetch.
    train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
    val_ds = val_ds.prefetch(tf.data.AUTOTUNE)
    test_ds = test_ds.prefetch(tf.data.AUTOTUNE)
    print("[OK] Augmentation solo en train_ds; prefetch aplicado al final en los tres pipelines.")


## 7. CNN Baseline simple (Student 2 en la futura escalera de KD)

Arquitectura ligera final (fijada 2026-09-22, ver `docs/decisions_log.md`): cuatro bloques Conv/Pool
(16/32/64/64 filtros) mantienen el modelo por debajo del presupuesto de <1M de parámetros,
conservando `Flatten` (requerimiento académico explícito).


In [ ]:
from neurodriver_cnn.models.baseline import build_baseline_cnn, compile_baseline, build_baseline_callbacks
from neurodriver_cnn.evaluation.metrics import count_parameters

training_config = load_training_config(PROJECT_ROOT)

baseline_model = build_baseline_cnn(
    input_shape=IMAGE_SIZE + (3,), dropout=training_config["baseline"]["dropout"]
)
compile_baseline(baseline_model, learning_rate=training_config["baseline"]["learning_rate"])
baseline_model.summary()

baseline_params = count_parameters(baseline_model)
print(baseline_params)
assert baseline_params["total_params"] < 1_000_000, (
    f"El Baseline CNN (Student 2) debe mantenerse ligero (<1M params), se obtuvo {baseline_params['total_params']}"
)
print(f"[OK] El Baseline CNN es ligero: {baseline_params['total_params']:,} parámetros totales (<1,000,000).")


## 8. Callbacks + entrenamiento del baseline

In [ ]:
callbacks = build_baseline_callbacks(
    patience_es=training_config["callbacks"]["early_stopping_patience"],
    patience_lr=training_config["callbacks"]["reduce_lr_patience"],
)

BASELINE_STATUS = "PENDING"
if train_ds is not None and val_ds is not None:
    # Una corrida corta es aceptable para obtener métricas preliminares honestas; NO es una
    # corrida de entrenamiento de producción. Los epochs se incrementan deliberadamente, no de forma automática.
    history = baseline_model.fit(
        train_ds, validation_data=val_ds, epochs=training_config["baseline"]["epochs"], callbacks=callbacks
    )
    BASELINE_STATUS = "DONE (preliminary short run)"
else:
    print("PENDING: se omite el entrenamiento del baseline, todavía no hay datos disponibles.")

print(f"Estado del baseline: {BASELINE_STATUS}")


In [ ]:
# Muestra explícitamente las curvas de entrenamiento registradas (loss/val_loss, binary_accuracy/val_binary_accuracy).
if BASELINE_STATUS.startswith("DONE"):
    print("Claves del history:", list(history.history.keys()))
    for key in ["loss", "val_loss", "binary_accuracy", "val_binary_accuracy"]:
        if key in history.history:
            print(f"  {key}: {history.history[key][-1]:.4f} (última época)")

    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(history.history["loss"], label="loss")
    axes[0].plot(history.history["val_loss"], label="val_loss")
    axes[0].set_title("Pérdida (Loss)")
    axes[0].legend()
    axes[1].plot(history.history["binary_accuracy"], label="binary_accuracy")
    axes[1].plot(history.history["val_binary_accuracy"], label="val_binary_accuracy")
    axes[1].set_title("Exactitud binaria (Binary accuracy)")
    axes[1].legend()
    plt.show()
else:
    print("PENDING: todavía no hay historial de entrenamiento para mostrar.")


## 9. Evaluación del baseline

Usa `labels_from_logits` para aplicar el threshold a las dos probabilidades sigmoid y derivar la
etiqueta ADAS de 4 estados; luego reporta tanto las métricas binarias por target (vehicle/pedestrian)
como las métricas de Precision/Recall/F1-Score y la matriz de confusión de 4 estados sobre
CLEAR/VEHICLE/PEDESTRIAN/MIXED — nunca solo Accuracy.


In [ ]:
from neurodriver_cnn.evaluation.metrics import (
    CLASS_NAME_TO_INDEX, classification_metrics, confusion_matrix, evaluate_motorcycle_subset,
    labels_from_logits, multilabel_binary_metrics,
)
from neurodriver_cnn.labeling.frame_labels import derive_label
import numpy as np

LABEL_THRESHOLD = load_training_config(PROJECT_ROOT)["evaluation"]["label_threshold"]


def evaluate_multilabel_model(model, dataset, threshold=LABEL_THRESHOLD):
    """Rutina de evaluación compartida: métricas binarias por target + métricas/matriz de confusión del 4-state derivado."""
    y_true = np.concatenate([y.numpy() for _, y in dataset])  # (N, 2): [has_vehicle, has_pedestrian]
    logits = model.predict(dataset)  # (N, 2): [vehicle_logit, pedestrian_logit]

    pred = labels_from_logits(logits[:, 0], logits[:, 1], threshold=threshold)
    true_labels = np.array(
        [CLASS_NAME_TO_INDEX[derive_label(bool(v), bool(p))] for v, p in y_true.astype(bool)]
    )

    print("Métricas binarias por target (vehicle, pedestrian):")
    print(multilabel_binary_metrics(y_true[:, 0], y_true[:, 1], pred["has_vehicle"], pred["has_pedestrian"]))
    print("\nMétricas del 4-state derivado (CLEAR/VEHICLE/PEDESTRIAN/MIXED) — Precision/Recall/F1-Score:")
    print(classification_metrics(true_labels, pred["label"]))
    print("\nMatriz de confusión (filas=real, columnas=predicho, orden CLEAR/VEHICLE/PEDESTRIAN/MIXED):")
    print(confusion_matrix(true_labels, pred["label"]))
    return true_labels, pred["label"]


if test_ds is not None and BASELINE_STATUS.startswith("DONE"):
    evaluate_multilabel_model(baseline_model, test_ds)
else:
    print("PENDING: la evaluación del baseline requiere un baseline entrenado y datos de TEST.")


## 10. Construcción del MobileNetV2 Student (Student 1)

In [ ]:
from neurodriver_cnn.models.mobilenetv2 import build_mobilenetv2_student, compile_student

student_model, student_base_model = build_mobilenetv2_student(
    input_shape=IMAGE_SIZE + (3,), dropout=training_config["mobilenetv2"]["dropout"], freeze_backbone=True
)
compile_student(student_model, learning_rate=training_config["mobilenetv2"]["head_learning_rate"])
student_model.summary()
print(count_parameters(student_model))


## 11. Smoke test de forward-pass de MobileNetV2

In [ ]:
MOBILENET_SMOKE_TEST_STATUS = "PENDING"
if train_ds is not None:
    for images, targets in train_ds.take(1):
        logits = student_model.predict(images, verbose=0)
        assert logits.shape == (images.shape[0], 2), logits.shape  # [vehicle_logit, pedestrian_logit]
        assert not np.isnan(logits).any(), "NaN en los logits del Student"
        MOBILENET_SMOKE_TEST_STATUS = "DONE"
        print(f"[OK] logits shape={logits.shape}, sin NaNs.")
else:
    print("PENDING: el smoke test de forward-pass requiere datos reales (un batch real), no tensores sintéticos.")

print(f"Estado del smoke test de MobileNetV2: {MOBILENET_SMOKE_TEST_STATUS}")


## 12. Entrenar MobileNetV2 base (backbone congelado)

Se ejecuta después de terminar el baseline (Secciones 7-9). Mismo pipeline `train_ds`/`val_ds`,
backbone de ImageNet **congelado** (entrenamiento solo del head), `BinaryCrossentropy(from_logits=True)`
— replica exactamente el loop de entrenamiento del baseline. Una corrida corta es aceptable para
obtener métricas preliminares honestas; NO es una corrida de producción. Usa
`training_config["mobilenetv2"]["head_epochs"]` / `["head_learning_rate"]`.

**El entrenamiento del ResNet50 Teacher y la Knowledge Distillation explícitamente NO se ejecutan**
— ver las Secciones 14-16.


In [ ]:
mobilenetv2_callbacks = build_baseline_callbacks(
    patience_es=training_config["callbacks"]["early_stopping_patience"],
    patience_lr=training_config["callbacks"]["reduce_lr_patience"],
)

MOBILENETV2_TRAINING_STATUS = "PENDING"
if train_ds is not None and val_ds is not None:
    mobilenetv2_history = student_model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=training_config["mobilenetv2"]["head_epochs"],
        callbacks=mobilenetv2_callbacks,
    )
    MOBILENETV2_TRAINING_STATUS = "DONE (frozen-backbone head training, preliminary short run)"
    print("Claves del history:", list(mobilenetv2_history.history.keys()))
else:
    print("PENDING: se omite el entrenamiento base de MobileNetV2, todavía no hay datos disponibles.")

print(f"Estado del entrenamiento base de MobileNetV2: {MOBILENETV2_TRAINING_STATUS}")


## 13. Evaluación base de MobileNetV2

In [ ]:
if test_ds is not None and MOBILENETV2_TRAINING_STATUS.startswith("DONE"):
    evaluate_multilabel_model(student_model, test_ds)
else:
    print("PENDING: la evaluación de MobileNetV2 requiere un modelo entrenado y datos de TEST.")


## 14. Construcción del ResNet50 Teacher (solo arquitectura, no entrenado)

Futuro Teacher para Knowledge Distillation. Mismo espacio de salida de dos logits que ambos Students,
por lo que no se necesita una capa adaptadora para comparar los logits de Teacher/Student más
adelante (ver `docs/teacher_student_contract.md`). **No se entrena en esta fase** — solo
construcción + conteo de parámetros + un smoke test de forward-pass.


In [ ]:
from neurodriver_cnn.models.resnet50_teacher import build_resnet50_teacher, compile_teacher

RESNET50_STATUS = "PENDING"
if train_ds is not None:
    teacher_model, teacher_base_model = build_resnet50_teacher(
        input_shape=IMAGE_SIZE + (3,),
        dropout=training_config["resnet50_teacher"]["dropout"],
        freeze_backbone=True,
    )
    compile_teacher(teacher_model, learning_rate=training_config["resnet50_teacher"]["head_learning_rate"])
    print(count_parameters(teacher_model))

    for images, targets in train_ds.take(1):
        teacher_logits = teacher_model.predict(images, verbose=0)
        assert teacher_logits.shape == (images.shape[0], 2), teacher_logits.shape
        assert not np.isnan(teacher_logits).any(), "NaN en los logits del Teacher"
        RESNET50_STATUS = "DONE (architecture + smoke test only, not trained)"
        print(f"[OK] logits shape del Teacher={teacher_logits.shape}, sin NaNs.")
else:
    print("PENDING: el smoke test del ResNet50 Teacher requiere datos reales.")

print(f"Estado del ResNet50 Teacher: {RESNET50_STATUS}")


## 15. Celdas de fine-tuning futuro (preparadas, no ejecutadas)

Procedimiento de fine-tuning una vez que el entrenamiento supervisado del head sobre BDD sea sólido
(aplica a ambos Students):

1. entrenar el head con el backbone congelado (Secciones 7-13, ya realizado arriba);
2. `unfreeze_for_fine_tuning(student_base_model, unfreeze_from_layer=...)`;
3. recompilar con un learning rate bajo (`training_config["mobilenetv2"]["fine_tune_learning_rate"]`, ~1e-5);
4. hacer fine-tuning con cuidado; las capas BatchNormalization se mantienen congeladas (modo
   inferencia) incluso al descongelar, ya que los batches pequeños de fine-tuning de BDD no son
   suficientemente representativos para actualizar de forma segura las estadísticas acumuladas de
   BatchNorm;
5. mantener el fine-tuning en el dominio fuente de BDD y el futuro fine-tuning en el dominio
   objetivo colombiano como **experimentos separados y comparables** (ver
   `docs/colombian_domain_strategy.md`).


In [ ]:
from neurodriver_cnn.models.mobilenetv2 import unfreeze_for_fine_tuning

FINE_TUNING_STATUS = "PENDING (not executed in Phase 1)"
# Ejemplo de lo que se ejecutará en la Fase 2 — intencionalmente no ejecutado aquí:
# unfreeze_for_fine_tuning(student_base_model, unfreeze_from_layer=training_config["mobilenetv2"]["fine_tune_unfreeze_from_layer"])
# compile_student(student_model, learning_rate=training_config["mobilenetv2"]["fine_tune_learning_rate"])
# student_model.fit(train_ds, validation_data=val_ds, epochs=training_config["mobilenetv2"]["fine_tune_epochs"], callbacks=callbacks)
print(FINE_TUNING_STATUS)


## 16. Preparación para Knowledge Distillation

- `student_model` (MobileNetV2), `baseline_model` (CNN ligera) y `teacher_model` (ResNet50) exponen
  todos la **misma salida de dos logits** (`vehicle_logit`, `pedestrian_logit`) directamente — sin
  Softmax, sin necesidad de un modelo de pre-activación separado, tal como se requiere para una
  futura pérdida de distillation por target (`docs/teacher_student_contract.md`).
- Todavía no existe entrenamiento real del Teacher, logits del Teacher, ni pérdida de KD —
  `configs/training_config.json` mantiene `distillation.enabled = false`.
- Ver `src/neurodriver_cnn/distillation/README.md` para los componentes planeados de pérdida de
  distillation / `Distiller` (calculados por target binario, no sobre un softmax de 4 vías).
- **Aquí no se realiza ninguna KD simulada.**


## 17. Resumen de estado

| Componente | Estado |
|---|---|
| Configuración de entorno/semillas | DONE |
| Carga y validación del manifest | DONE si existe `experiment_manifest.csv`, si no PENDING |
| Pipeline de tf.data (targets multi-label) + Data Augmentation solo en train + prefetch como último paso | DONE si hay datos disponibles, si no PENDING |
| El Baseline CNN (Student 2) es ligero (<1M params) | DONE — verificado en la Sección 7 (863,522 parámetros totales) |
| Entrenamiento del Baseline CNN (Student 2) | ver `BASELINE_STATUS` arriba |
| Evaluación del baseline (por target + Precision/Recall/F1-Score/matriz de confusión del 4-state derivado) | PENDING salvo que el baseline esté entrenado y haya datos de TEST disponibles |
| Construcción del MobileNetV2 Student (Student 1) | DONE |
| Smoke test de forward-pass de MobileNetV2 | ver `MOBILENET_SMOKE_TEST_STATUS` arriba |
| Entrenamiento base de MobileNetV2 (backbone congelado) | ver `MOBILENETV2_TRAINING_STATUS` arriba |
| Evaluación base de MobileNetV2 | PENDING salvo que esté entrenado y haya datos de TEST disponibles |
| Construcción y smoke test del ResNet50 Teacher | ver `RESNET50_STATUS` arriba |
| Entrenamiento del ResNet50 Teacher | NO ejecutado (Fase 2+) |
| Fine-tuning | PENDING (Fase 2+) |
| Knowledge Distillation | PENDING (Fase 2+, requiere un ResNet50 Teacher entrenado) |
